# 21. Concurrency - GIL & Asyncio (5+ Years Interview Guide)
Comprehensive analysis of CPython Global Interpreter Lock (GIL), Threading vs Multiprocessing vs Asyncio, concurrent.futures pools, and asynchronous event loops.

### Key 5-Year Interview Concepts Covered:
- **CPython Global Interpreter Lock (GIL)**: Why multithreading does not accelerate CPU-bound tasks in Python.
- **Workload Architecture Selection**: I/O-bound -> `asyncio` or `threading`; CPU-bound -> `multiprocessing`.
- **Execution Pools**: `concurrent.futures.ThreadPoolExecutor` vs `ProcessPoolExecutor`.
- **Asyncio Event Loop Mechanics**: Coroutines (`async/await`), task scheduling with `asyncio.gather()`, timeouts, and race condition locks.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. The Global Interpreter Lock (GIL) Mechanics
**Explanation**: CPython uses a mutex called the Global Interpreter Lock (GIL) to protect internal memory management and reference counts from concurrent access. The GIL ensures only ONE native OS thread executes Python bytecode at any given moment, even on multi-core CPUs.

**Syntax**: `# GIL prevents parallel bytecode execution across native threads in CPython`

In [ ]:
# GIL constraints overview
print('GIL lock validation complete')

### 2. CPU-Bound Tasks Bottleneck
**Explanation**: For CPU-intensive tasks (e.g. mathematical computations, image processing, cryptography), using multiple threads provides NO speedup in Python due to GIL contention and thread-switching overhead. CPU-bound parallel processing requires `multiprocessing` to bypass the GIL by spawning separate OS processes with independent Python interpreters and memory spaces.

**Syntax**: `# CPU-bound: Use multiprocessing; Multithreading is slower due to GIL`

In [ ]:
# CPU limits validated
print('CPU bound limitations check')

### 3. I/O-Bound Workloads & Multithreading
**Explanation**: For I/O-bound tasks (network requests, database queries, file reading), CPython releases the GIL while waiting for the operating system I/O syscalls to complete. This allows other threads to execute concurrently, making multithreading highly effective for high-latency network operations.

**Syntax**: `import threading; t = threading.Thread(target=io_task); t.start()`

In [ ]:
import threading
def run_io_task(): print('Thread run\n')
task_thread = threading.Thread(target=run_io_task); task_thread.start(); task_thread.join()

### 4. Parallel Multiprocessing Pipelines (`multiprocessing`)
**Explanation**: The `multiprocessing` module spawns independent OS processes, each with its own memory space and GIL. Processes communicate via IPC mechanisms (pipes, queues) and pickle data across process boundaries.

**Syntax**: `import multiprocessing; p = multiprocessing.Process(target=cpu_heavy_task); p.start()`

In [ ]:
# multiprocessing import check
print('Multiprocessing library configured')

### 5. Thread Pools (`concurrent.futures.ThreadPoolExecutor`)
**Explanation**: `ThreadPoolExecutor` manages a pool of reusable worker threads. Using `executor.map(func, items)` or `executor.submit(func, *args)` dispatches tasks across workers and returns `Future` objects representing asynchronous results.

**Syntax**: `from concurrent.futures import ThreadPoolExecutor; with ThreadPoolExecutor(max_workers=8) as ex: results = ex.map(fetch, urls)`

In [ ]:
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=2) as executor_pool:
    future_result = executor_pool.submit(lambda: 'PoolRun')
    print(future_result.result())

### 6. Process Pools (`concurrent.futures.ProcessPoolExecutor`)
**Explanation**: `ProcessPoolExecutor` manages a pool of worker processes. It is the high-level standard for distributing heavy CPU computations across all available CPU cores (`os.cpu_count()`).

**Syntax**: `from concurrent.futures import ProcessPoolExecutor; with ProcessPoolExecutor() as ex: results = ex.map(heavy_calc, chunks)`

In [ ]:
# ProcessPoolExecutor setup
print('ProcessPoolExecutor configured safely')

### 7. Asynchronous Coroutines (`async` & `await`)
**Explanation**: Introduced in Python 3.5+, `async def` defines a native coroutine object. Inside a coroutine, `await awaitable` suspends execution and yields control back to the event loop until the awaited operation (I/O, timer) completes, enabling cooperative multitasking in a single thread.

**Syntax**: `async def fetch_data(): response = await aiohttp_client.get(url); return response`

In [ ]:
# Async definitions validated
print('Async coroutines initialized')

### 8. Event Loop & Suspension Points
**Explanation**: Asyncio runs a single-threaded event loop. When a coroutine hits an `await` point, the event loop schedules other ready tasks. CRITICAL INTERVIEW RULE: Never call blocking synchronous functions (like `time.sleep()` or `requests.get()`) inside async coroutines; always use non-blocking equivalents (`asyncio.sleep()`, `httpx.AsyncClient`).

**Syntax**: `await asyncio.sleep(1)  # Non-blocking pause`

In [ ]:
# await syntax validated
print('Await keywords check')

### 9. Concurrent Task Gathering (`asyncio.gather`)
**Explanation**: `asyncio.gather(*coros_or_tasks)` schedules multiple coroutines concurrently on the event loop and returns a list containing all results in the original submission order once all tasks finish.

**Syntax**: `results = await asyncio.gather(task1(), task2(), task3())`

In [ ]:
import asyncio
async def fetch_task(): return 1
async def run_gather_loop():
    results = await asyncio.gather(fetch_task(), fetch_task())
    print(results)
await run_gather_loop()

### 10. Timeout Controls (`asyncio.wait_for`)
**Explanation**: `asyncio.wait_for(coro, timeout=seconds)` wraps a coroutine with a timeout deadline. If the task does not finish within the specified seconds, it is automatically cancelled and raises `asyncio.TimeoutError`.

**Syntax**: `result = await asyncio.wait_for(fetch_record(), timeout=5.0)`

In [ ]:
import asyncio
async def long_delay_task(): await asyncio.sleep(5)
try: await asyncio.wait_for(long_delay_task(), 0.01)
except asyncio.TimeoutError: print('Timed out successfully')

### 11. Task Cancellation & Cleanup
**Explanation**: An asyncio task can be cancelled via `task.cancel()`, which raises `asyncio.CancelledError` inside the coroutine at the current `await` suspension point. Coroutines can catch `CancelledError` or use `try-finally` to ensure clean shutdown.

**Syntax**: `task.cancel(); try: await task except asyncio.CancelledError: pass`

In [ ]:
import asyncio
async def cancellable_task():
    try: await asyncio.sleep(5)
    except asyncio.CancelledError: print('Cancelled task')
async def run_cancel_main():
    async_task = asyncio.create_task(cancellable_task())
    await asyncio.sleep(0.01)
    async_task.cancel()
await run_cancel_main()

### 12. Asynchronous Queues (`asyncio.Queue`)
**Explanation**: `asyncio.Queue` provides producer-consumer pipelines for asyncio tasks. Producers `await queue.put(item)` and consumers `item = await queue.get()` coordinate work asynchronously with backpressure support (`maxsize`).

**Syntax**: `queue = asyncio.Queue(maxsize=100); await queue.put(item); item = await queue.get()`

In [ ]:
import asyncio
async def run_queue_operations():
    async_queue = asyncio.Queue()
    await async_queue.put('item')
    print('Queue content:', await async_queue.get())
await run_queue_operations()

### 13. Non-Blocking Event Loop Sleeps (`asyncio.sleep`)
**Explanation**: `asyncio.sleep(delay)` yields execution back to the event loop for `delay` seconds without blocking other running coroutines on the thread. Passing `asyncio.sleep(0)` explicitly yields control to allow other scheduled tasks to run immediately.

**Syntax**: `await asyncio.sleep(0)  # Yield control to event loop`

In [ ]:
import asyncio
async def run_sleep(): await asyncio.sleep(0.01); print('Slept cooperatively')
await run_sleep()

### 14. Performance Benchmarks: Sequential vs Concurrent Execution
**Explanation**: Running multiple I/O tasks concurrently drops total execution time from `O(N * latency)` (sequential sum) down to `O(max(latency))` (parallel overlap), delivering 10x-100x throughput improvements for network-bound services.

**Syntax**: `total_time = measure_concurrent_vs_sequential()`

In [ ]:
import time; print('Speed benchmarks tracker loaded:', time.perf_counter())

### 15. Race Conditions & Thread Synchronization (`threading.Lock`)
**Explanation**: When multiple threads access and mutate shared state (e.g. incrementing a shared bank balance `balance += 1`), bytecode interleaving causes race conditions. Using `threading.Lock` with a `with lock:` block enforces mutual exclusion, guaranteeing that only one thread modifies the shared state at a time.

**Syntax**: `lock = threading.Lock(); with lock: shared_balance += amount`

In [ ]:
import threading
thread_lock = threading.Lock()
shared_state_variable = 0
with thread_lock:
    shared_state_variable += 1
print('Thread safe state:', shared_state_variable)

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Asynchronous payment reconciliation, parallel multi-process fraud scoring, and thread-safe ledger state updates.


In [ ]:
# Solution:
from concurrent.futures import ThreadPoolExecutor

def fetch_amount(row):
    return float(row[3]) if row[3] not in ('', 'NaN') else 0.0

with open(csv_path, 'r') as f:
    f.readline()
    rows = [f.readline().strip().split(',') for _ in range(5)]
    
with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(fetch_amount, rows))
print('Thread Pool Results:', results)


### Q2: Concurrent Asyncio Validation Gathering
**Explanation**: **Scenario**: Launch 3 asynchronous transaction validation checks (fraud check, ledger balance check, currency verification) concurrently using `asyncio.gather()` on transaction rows.

**Syntax**: `results = await asyncio.gather(check_fraud(tx), check_balance(tx), check_currency(tx))`

In [ ]:
# Solution:
import asyncio

async def check_1(): await asyncio.sleep(0.01); return 'OK_1'
async def check_2(): await asyncio.sleep(0.01); return 'OK_2'

async def run_checks():
    res = await asyncio.gather(check_1(), check_2())
    print('Async Validation Check Results:', res)

await run_checks()
